# 17 — Streaming Speech-to-Speech

## Objective

In this notebook, we will explore the core ideas behind low-latency streaming speech-to-speech systems inspired by architectures such as Moshi and Hibiki.

Instead of using a classical sequential pipeline:

Audio
→ ASR
→ LLM
→ TTS
→ Audio

we will simulate a streaming architecture where audio is processed frame by frame using discrete audio tokens.

We will focus on:

- 80 ms streaming audio frames;
- discrete audio token simulation;
- parallel user and assistant audio streams;
- inner-monologue text tokens;
- frame-by-frame processing;
- streaming latency.

## Streaming architecture

A full-duplex speech-to-speech system can process incoming and outgoing audio continuously.

Conceptually:

User Audio Stream
→ Audio Codec
→ Discrete Audio Tokens
→ Transformer
→ Assistant Audio Tokens
→ Audio Decoder
→ Assistant Audio Stream

At the same time, the model may generate text tokens representing an internal semantic stream.

## Simulate 80 ms audio frames

Streaming speech-to-speech systems process audio continuously in short frames.

In this experiment, we will use 80 ms frames, similar to the framing used in modern low-latency speech models.

Each frame will be converted into a small set of simulated discrete audio tokens.

In [1]:
import numpy as np
import time

sample_rate = 16000
frame_duration = 0.080
frame_size = int(sample_rate * frame_duration)

print("Frame duration:", frame_duration * 1000, "ms")
print("Samples per frame:", frame_size)

Frame duration: 80.0 ms
Samples per frame: 1280


## Simulate discrete audio tokens

A neural audio codec converts each audio frame into a small set of discrete token IDs.

To keep this notebook lightweight, we will simulate this process instead of running a real codec.

For each 80 ms frame, we will generate 8 discrete token IDs representing 8 codec codebooks.

In [2]:
num_codebooks = 8
codebook_size = 1024

def simulate_codec_tokens():
    return np.random.randint(
        0,
        codebook_size,
        size=num_codebooks
    )

tokens = simulate_codec_tokens()

print("Simulated audio tokens:")
print(tokens)

print("\nNumber of codebooks:", len(tokens))

Simulated audio tokens:
[576 590 823 372 558  46 561  77]

Number of codebooks: 8


## Simulate parallel user and assistant streams

A full-duplex speech-to-speech system processes the user's incoming audio while also generating the assistant's outgoing audio.

We will simulate three parallel streams:

- user audio tokens;
- assistant audio tokens;
- inner-monologue text tokens.

In [3]:
num_frames = 12

user_stream = []
assistant_stream = []
text_stream = []

text_tokens = [
    "hello",
    "I",
    "can",
    "help",
    "with",
    "your",
    "request",
    "today"
]

for i in range(num_frames):
    user_tokens = simulate_codec_tokens()
    assistant_tokens = simulate_codec_tokens()

    user_stream.append(user_tokens)
    assistant_stream.append(assistant_tokens)

    if i < len(text_tokens):
        text_stream.append(text_tokens[i])
    else:
        text_stream.append("...")

print("Frame 0")
print("User tokens:", user_stream[0])
print("Assistant tokens:", assistant_stream[0])
print("Text token:", text_stream[0])

Frame 0
User tokens: [460  49 210  54 282 783 274 930]
Assistant tokens: [627 705 641 420 919 787 712 621]
Text token: hello


In [4]:
for i in range(5):
    print(f"\nFrame {i}")
    print("User:", user_stream[i])
    print("Assistant:", assistant_stream[i])
    print("Text:", text_stream[i])


Frame 0
User: [460  49 210  54 282 783 274 930]
Assistant: [627 705 641 420 919 787 712 621]
Text: hello

Frame 1
User: [ 989  338  229 1003  778  367  164   45]
Assistant: [1004  237  310  352  200  581  298  946]
Text: I

Frame 2
User: [438 667 290 955 687 347 377 517]
Assistant: [168 830 437 974 733 552 584 735]
Text: can

Frame 3
User: [315 283 125 830 761 914 779 675]
Assistant: [363 865 109 478 349 682 297 243]
Text: help

Frame 4
User: [300 367 796 450 153 220 668 498]
Assistant: [149 101 154 808 284 577 177 591]
Text: with


## Simulate streaming processing

In a real-time speech-to-speech system, each audio frame must be processed before the next frame arrives.

With 80 ms frames, the processing time should ideally remain below 80 ms to avoid accumulating latency.

We will simulate a lightweight processing step for each frame and measure the latency.

In [5]:
processing_times = []

for i in range(num_frames):
    start_time = time.perf_counter()

    user_tokens = user_stream[i]
    assistant_tokens = assistant_stream[i]
    text_token = text_stream[i]

    # Simulated processing
    combined_value = (
        np.mean(user_tokens)
        + np.mean(assistant_tokens)
    )

    end_time = time.perf_counter()

    processing_time_ms = (end_time - start_time) * 1000
    processing_times.append(processing_time_ms)

    print(
        f"Frame {i:02d} | "
        f"Text: {text_token:8s} | "
        f"Latency: {processing_time_ms:.4f} ms"
    )

Frame 00 | Text: hello    | Latency: 0.0755 ms
Frame 01 | Text: I        | Latency: 0.0305 ms
Frame 02 | Text: can      | Latency: 0.0254 ms
Frame 03 | Text: help     | Latency: 0.0248 ms
Frame 04 | Text: with     | Latency: 0.0310 ms
Frame 05 | Text: your     | Latency: 0.0251 ms
Frame 06 | Text: request  | Latency: 0.0250 ms
Frame 07 | Text: today    | Latency: 0.0245 ms
Frame 08 | Text: ...      | Latency: 0.0242 ms
Frame 09 | Text: ...      | Latency: 0.0242 ms
Frame 10 | Text: ...      | Latency: 0.0243 ms
Frame 11 | Text: ...      | Latency: 0.0243 ms


In [6]:
print("\nAverage frame latency:",
      round(np.mean(processing_times), 4),
      "ms")

print("Maximum frame latency:",
      round(np.max(processing_times), 4),
      "ms")

print("Frame duration:",
      frame_duration * 1000,
      "ms")


Average frame latency: 0.0299 ms
Maximum frame latency: 0.0755 ms
Frame duration: 80.0 ms


### Streaming latency result

The simulated processing latency is much lower than the 80 ms frame duration.

Results:

- Average latency: 0.0782 ms
- Maximum latency: 0.559 ms
- Frame duration: 80 ms

This means that the simulated processing can easily keep up with the incoming audio stream without accumulating delay.

In a real speech-to-speech model, the processing would be significantly more expensive because neural codecs and Transformers must run for every frame.

## Real-Time Factor

The Real-Time Factor compares the total processing time with the duration of the audio stream.

A system can operate in real time when:

RTF < 1

The smaller the RTF, the more computational margin the system has.

In [7]:
total_processing_time = sum(processing_times) / 1000  # ms → seconds
stream_duration = num_frames * frame_duration

rtf = total_processing_time / stream_duration

print("Stream duration:", round(stream_duration, 3), "seconds")
print("Total processing time:", round(total_processing_time, 6), "seconds")
print("Real-Time Factor:", round(rtf, 6))

Stream duration: 0.96 seconds
Total processing time: 0.000359 seconds
Real-Time Factor: 0.000374


### Real-Time Factor result

The simulated streaming system achieved a Real-Time Factor of approximately:

0.000977

Since RTF < 1, the system can process the stream faster than the audio arrives.

The very low RTF is expected because this notebook uses lightweight symbolic operations instead of a real neural codec and Transformer model.

In a real Moshi- or Hibiki-style system, the computational cost would be much higher.

## Simulate full-duplex interaction

A full-duplex speech-to-speech system can listen to the user while simultaneously generating assistant audio.

This differs from a traditional turn-based pipeline where one side must stop before the other can begin.

We will simulate overlapping user and assistant activity across multiple audio frames.

In [8]:
user_active = [
    False, False,
    True, True, True, True,
    False, False,
    True, True,
    False, False
]

assistant_active = [
    False, True,
    True, True, True,
    False, False, True,
    True, True,
    True, False
]

for i in range(num_frames):

    user_state = "SPEAKING" if user_active[i] else "silent"
    assistant_state = "SPEAKING" if assistant_active[i] else "silent"

    overlap = user_active[i] and assistant_active[i]

    print(
        f"Frame {i:02d} | "
        f"User: {user_state:8s} | "
        f"Assistant: {assistant_state:8s} | "
        f"Overlap: {overlap}"
    )

Frame 00 | User: silent   | Assistant: silent   | Overlap: False
Frame 01 | User: silent   | Assistant: SPEAKING | Overlap: False
Frame 02 | User: SPEAKING | Assistant: SPEAKING | Overlap: True
Frame 03 | User: SPEAKING | Assistant: SPEAKING | Overlap: True
Frame 04 | User: SPEAKING | Assistant: SPEAKING | Overlap: True
Frame 05 | User: SPEAKING | Assistant: silent   | Overlap: False
Frame 06 | User: silent   | Assistant: silent   | Overlap: False
Frame 07 | User: silent   | Assistant: SPEAKING | Overlap: False
Frame 08 | User: SPEAKING | Assistant: SPEAKING | Overlap: True
Frame 09 | User: SPEAKING | Assistant: SPEAKING | Overlap: True
Frame 10 | User: silent   | Assistant: SPEAKING | Overlap: False
Frame 11 | User: silent   | Assistant: silent   | Overlap: False


### Full-duplex result

The simulation shows that user and assistant activity can overlap across the same audio frames.

When both streams are active at the same time:

User: SPEAKING
Assistant: SPEAKING
Overlap: True

This represents the core idea of full-duplex speech interaction.

Unlike a traditional turn-based pipeline, the system does not require the assistant to completely stop listening while it is speaking.

## Conclusion

In this notebook, we explored the main ideas behind low-latency streaming speech-to-speech systems inspired by Moshi and Hibiki.

### What we explored

- 80 ms streaming audio frames;
- simulated discrete audio codec tokens;
- parallel user and assistant audio streams;
- inner-monologue text tokens;
- frame-by-frame streaming processing;
- latency measurement;
- Real-Time Factor;
- full-duplex interaction.

### Final architecture

User Audio Stream
→ Audio Codec
→ Discrete Audio Tokens
→ Streaming Transformer
→ Assistant Audio Tokens
→ Audio Decoder
→ Assistant Audio Stream

### Key takeaway

Streaming speech-to-speech systems process audio continuously instead of waiting for complete turns.

Full-duplex architectures allow the model to listen and speak at the same time, reducing conversational latency and enabling more natural interruptions and interactions.